In [ ]:
import os, cv2, json, glob
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import torch
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import sys, gc, ast
sys.path.append("..")
from Dataset.index import Normalization
from Network.index import ModelNetwork
from Transforms.index import Transforms
from utils.index import showTile, faciesCmaps

In [ ]:
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

gc.collect()
print(torch.__version__)              # versão do PyTorch
print(torch.cuda.is_available())      # True se detectou a GPU
print(torch.cuda.get_device_name(0))  # nome da GPU

# MODELO
- Base Path é a pasta dos modelos: '../Model/Backup', '../Documents/Backups/Backup1' ou '../Documents/Backups/Backup2'
- Dataset=None é dataset do modelo; um nome avalia no teste dele sem os tiles do treino

In [ ]:
BASE_PATH = '../Model/Backup'    
DATASET   = None                            

In [ ]:
models    = sorted(os.listdir(BASE_PATH), key=lambda name: int(name.split('_')[-1]))
df_models = []

for model in models:
    info     = json.loads(open(f'{BASE_PATH}/{model}/info.json', 'r').read())
    n_images = info.get('division', info['processing']).get('n_images', {})
    
    df_models.append({
        'id': model,
        'network': info['model'].get('network'),
        'dataset': info['processing'].get('dataset'),
        'num_filters': info['processing'].get('num_filters'),
        'dropout': info['processing'].get('dropout'),
        'lr': info['processing'].get('lr'),
        'batch_size': info['processing'].get('batch_size'),
        'train_size': n_images.get('train'),
        'img_size': info['model'].get('img_size'),
        'test_iou': info['trainer'].get('test_iou'),
        'marlim_iou': info.get('marlim_iou', {}).get('mean'),
        'path': f'{BASE_PATH}/{model}',
    })

df_models = pd.DataFrame(df_models).sort_values(by='dataset')
df_models

In [ ]:
modelId = models[-1]   # o último treinado, que o Task/index.py avalia depois de cada treino; ou 'model_1' para fixar um modelo
baseDir = f'{BASE_PATH}/{modelId}'

modelInfo = json.loads(open(f'{baseDir}/info.json', 'r').read())
OPTIONS   = modelInfo.get('processing')
DIVISION  = modelInfo.get('division', OPTIONS)

print(DIVISION)
OPTIONS

In [ ]:
model_options = modelInfo.get('model')
print(model_options)

network   = ModelNetwork(**model_options)
modelData = torch.load(f'{baseDir}/model.pth', map_location=network.device)

network.model.load_state_dict(modelData['model'])
network.model.eval()

# SPLITS DOS DADOS

- O `TEST_SIZE` vem do `division` do `info.json` do modelo, gravado pelo `1 - Model` (backup de antes do `division` guardava o `n_images` e o `step` no `processing`, e o notebook lê de lá; o `test_size` era o 0.1 fixo do `1 - Model`): aqui não existe valor próprio, então não tem como treinar com um e testar com outro.
- O split é refeito igual ao do treino e as contagens têm que bater com o `n_images` gravado; se não baterem, a célula para.

In [ ]:
DATABASE = f'../Dataset/{OPTIONS.get("dataset")}/DataBase.csv'
df       = pd.read_csv(DATABASE)
IMG_SIZE = tuple(model_options.get('img_size'))
STEP     = DIVISION.get('step', OPTIONS.get('step'))

SCALING   = OPTIONS.get('scaling', Normalization.DEFAULT)
FORMATTED = Normalization.read(DATABASE)
if FORMATTED != SCALING:
    raise ValueError(f'{DATABASE} está com scaling={FORMATTED!r}, modelo treinou com scaling={SCALING!r}: rode o Format do dataset')

# E CORTADO COM O MESMO step: OUTRO step MUDA OS TILES E OS INLINES QUE O VOLUME INTEIRO MEDE
if int(df['step'].iloc[0]) != STEP:
    raise ValueError(f'{DATABASE} foi cortado com step={df["step"].iloc[0]} e o modelo treinou com step={STEP}: rode o Format do dataset')

print(f'{DATABASE}: {len(df)} tiles de shape {df["shape"].iloc[0]}, step {STEP}, janela {IMG_SIZE}')
df

In [ ]:
# BACKUP DE ANTES DO division NÃO GRAVAVA O test_size: O 1 - Model DO FACIES USAVA 0.1 FIXO (O n_images ABAIXO CONFERE)
TEST_SIZE = DIVISION.get('test_size', 0.1)
TEST_SIZE

In [ ]:
temp_df, test_df = train_test_split(df, test_size=TEST_SIZE, random_state=42)
n_images = {'total': len(df), 'test': len(test_df)}

if any(DIVISION['n_images'][key] != n_images[key] for key in n_images):
    raise ValueError(f'split diferente do treino: {n_images} aqui, {DIVISION["n_images"]} no info.json')

n_images

### Outro dataset

- Com `DATASET`, o teste é o split desse dataset com o mesmo `TEST_SIZE`, sem os tiles que o modelo já viu no treino e na validação (os mixes apontam para os mesmos arquivos dos datasets de origem).

In [ ]:
if DATASET and DATASET != OPTIONS.get('dataset'):
    seen = set(temp_df.img_path)
    df   = pd.read_csv(f'../Dataset/{DATASET}/DataBase.csv')
    temp_df, test_df = train_test_split(df, test_size=TEST_SIZE, random_state=42)
    test_df = test_df[~test_df.img_path.isin(seen)]

print(f'{len(test_df)} tiles de teste')
test_df.img_path.str.split('/').str[-4].value_counts()

# DATASET

- Modelo treinado com `crop` no `augmentations` tem a janela menor que o tile: o tile vai inteiro e a predição é pela janela deslizante do `Transforms`, com o mesmo `overlap`, `mode` e lote do treino, a mesma conta do `test_iou` gravado.
- Sem `crop`, os blocos são da janela do modelo, como sempre.

In [ ]:
transforms = Transforms(OPTIONS.get('augmentations'), batch=OPTIONS.get('batch_size'))
print(f'recorte {transforms.window}: tile inteiro por janela deslizante (overlap {transforms.overlap}, {transforms.mode})' if transforms.window else f'sem recorte: blocos de {IMG_SIZE}')

In [ ]:
class PredictDataset(Dataset):
    def __init__(self, df, window):
        self.df     = df.reset_index(drop=True)
        self.shape  = np.load(self.df.img_path.iloc[0], mmap_mode='r').shape
        self.window = tuple(min(int(w), int(s)) for w, s in zip(window or self.shape, self.shape))
        self.starts = [self.getStarts(s, w) for s, w in zip(self.shape, self.window)]
        self.blocks = [(row, d, h, w) for row in range(len(self.df)) for d in self.starts[0] for h in self.starts[1] for w in self.starts[2]]

    # INICIO DE CADA BLOCO NUM EIXO; O ULTIMO ENCOSTA NO FIM DO VOLUME
    @staticmethod
    def getStarts(size, window):
        return sorted(set(list(range(0, size - window + 1, window)) + [size - window]))

    def __len__(self):
        return len(self.blocks)

    def __getitem__(self, index):
        fileIndex, d, h, w = self.blocks[index]
        row = self.df.iloc[fileIndex]
        wd, wh, ww = self.window

        img  = np.asarray(np.load(row.img_path,  mmap_mode='r')[d:d + wd, h:h + wh, w:w + ww], np.float32)
        mask = np.asarray(np.load(row.mask_path, mmap_mode='r')[d:d + wd, h:h + wh, w:w + ww], np.float32)

        img_tensor  = torch.tensor(img,  dtype=torch.float32).unsqueeze(0)
        mask_tensor = torch.tensor(mask, dtype=torch.long).unsqueeze(0)
        return (img_tensor, mask_tensor)


testDataset = PredictDataset(
    df=test_df,
    window=None if transforms.window else IMG_SIZE
)

# DATA LOADER

In [ ]:
NUM_WORKERS = 2
BATCH_SIZE  = int(OPTIONS.get('batch_size') or 1)

testLoader = DataLoader(
    testDataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True if torch.cuda.is_available() else False
)

# DADOS DE TESTE

- Voxels de todos os tiles de teste somados numa matriz de confusão `classes × classes` antes de dividir, a mesma conta do `test_iou` do treino.
- Cada classe é medida contra as outras (`tp` na diagonal, `fp` no resto da coluna, `fn` no resto da linha); a linha `mean` é a média das classes, que é o `test_iou`, e a acurácia dela é a global.

In [ ]:
def getConfusion(network, loader, amp=False):
    network.model.eval()
    classes = max(network.classes, 2)
    matrix  = torch.zeros(classes * classes, dtype=torch.int64, device=network.device)

    with torch.no_grad(), torch.autocast('cuda', enabled=amp):
        for imgs, masks in tqdm(loader, desc='Avaliando'):
            imgs, masks = imgs.to(network.device), masks.to(network.device)
            logits = transforms.infer(network.model, imgs)

            if network.multiclass:
                preds  = torch.argmax(logits, dim=1)
                target = masks.squeeze(1) if masks.dim() == 5 else masks
            else:
                preds  = (torch.sigmoid(logits) > 0.5).long()
                target = masks

            # LINHA É A CLASSE REAL, COLUNA A PREDITA
            matrix += torch.bincount((target * classes + preds).flatten(), minlength=classes * classes)

    return matrix.view(classes, classes).cpu().numpy()


confusion = getConfusion(network, testLoader, amp=OPTIONS.get('amp', False))
confusion

In [ ]:
tp = np.diag(confusion)
fp = confusion.sum(axis=0) - tp
fn = confusion.sum(axis=1) - tp
tn = confusion.sum() - tp - fp - fn

testTable = pd.DataFrame({
    'dataset':   DATASET or OPTIONS.get('dataset'),
    'classe':    [str(c) for c in range(len(tp))],
    'accuracy':  (tp + tn) / confusion.sum(),
    'precision': tp / np.maximum(tp + fp, 1),
    'recall':    tp / np.maximum(tp + fn, 1),
    'iou':       tp / np.maximum(tp + fp + fn, 1),
    'f1':        2 * tp / np.maximum(2 * tp + fp + fn, 1),
})

testTable.loc[len(testTable)] = {'dataset': DATASET or OPTIONS.get('dataset'), 'classe': 'mean', 'accuracy': tp.sum() / confusion.sum(), **testTable[['precision', 'recall', 'iou', 'f1']].mean().to_dict()}
print(f'IoU médio {testTable.iou.iloc[-1]:.4f} | test_iou gravado no treino {modelInfo["trainer"].get("test_iou"):.4f}')
testTable

# MATRIZ DE CONFUSÃO

In [ ]:
# MATRIZ DE CONFUSAO DOS VOXELS; A COR E A FRACAO DA CLASSE REAL (LINHA), O TEXTO TRAZ TAMBEM A CONTAGEM
def plotConfusion(matrix, title=None, save=None):
    rates = matrix / np.maximum(matrix.sum(axis=1, keepdims=True), 1)
    n     = len(matrix)

    fig, ax = plt.subplots(figsize=(max(10, 1.6 * n + 4), max(6, 1.2 * n + 2)))
    image = ax.imshow(rates, cmap='Blues', vmin=0, vmax=1)
    fig.colorbar(image, ax=ax)

    for i in range(n):
        for j in range(n):
            ax.text(j, i, f'{rates[i, j]:.1%}\n{matrix[i, j]:,} voxels', ha='center', va='center', fontsize=9 if n > 2 else 11, color='white' if rates[i, j] > 0.5 else 'black')

    ax.set_xticks(range(n))
    ax.set_yticks(range(n))
    ax.set_xlabel('Prediction'), ax.set_ylabel('Ground Truth')
    ax.set_title(title)
    if not save:
        return plt.show()
    plt.savefig(save, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close(fig)


plotConfusion(confusion, title=f'Confusion Matrix')

# PREDIÇÕES

In [ ]:
def plotPrediction(index, dataset=testDataset, network=network, alpha=0.5, savePath=None):
    img_tensor, mask_tensor = dataset[index]
    network.model.eval()

    with torch.no_grad():
        img_batch = img_tensor.unsqueeze(0).to(network.device)
        logits    = transforms.infer(network.model, img_batch)
        pred = torch.argmax(logits, dim=1).squeeze() if network.multiclass else (torch.sigmoid(logits) > 0.5).squeeze().int()

    img_np  = img_tensor.squeeze().cpu().numpy()
    mask_np = mask_tensor.squeeze().cpu().numpy()
    pred_np = pred.cpu().numpy()

    if network.multiclass:
        solid_cmap, cmap = faciesCmaps(network.classes)
        mx, my, mz = img_np.shape[0] // 2, img_np.shape[1] // 2, img_np.shape[2] // 2
        planes = lambda v: [np.array(v[mx, :, :]), np.rot90(np.array(v[:, :, mz]), -1), np.array(v[:, my, :])]
        ig, mg, pg = planes(img_np), planes(mask_np), planes(pred_np)

        fig, axes = plt.subplots(2, 3, figsize=(13, 8))
        for j in range(3):
            for r, layer in enumerate([mg, pg]):
                axes[r, j].imshow(ig[j], cmap='gray')
                axes[r, j].imshow(layer[j], cmap=cmap, vmin=0, vmax=network.classes - 1, alpha=alpha)
                axes[r, j].axis('off')

        from matplotlib.patches import Patch
        legend_elements = [Patch(facecolor=solid_cmap.colors[i], edgecolor='white', label=f'Classe {i}') for i in range(network.classes)]
        fig.legend(handles=legend_elements, loc='lower center', ncol=min(network.classes, 7), bbox_to_anchor=(0.5, 0.01), fontsize=11)

        plt.tight_layout(rect=[0, 0.05, 1, 1])
        if savePath:
            plt.savefig(savePath, bbox_inches='tight', dpi=200); plt.close(fig)
        else:
            plt.show()
        return

    imgNorm = cv2.normalize(img_np, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    imgRgb  = np.stack((imgNorm, imgNorm, imgNorm), axis=-1)
    overlay = imgRgb.copy()
    is_gt   = (mask_np > 0)
    is_pred = (pred_np > 0)
    inters  = (is_gt & is_pred)   # Acertos / Interseção

    overlay[is_gt]   = [0, 0, 255]  # Azul para o Ground Truth (Falhas reais não detectadas)
    overlay[is_pred] = [255, 0, 0]  # Vermelho para a Predição (Falsos alarmes)
    overlay[inters]  = [0, 255, 0]  # verde onde Predição e GT convergem (Acertos)

    result = (imgRgb * (1 - alpha) + overlay * alpha).astype(np.uint8)
    showTile(img=result, save=savePath)
        

for i in range(min(len(testDataset), 2)):
    plotPrediction(index=i, dataset=testDataset, network=network)

# PREDIÇÃO DO VOLUME INTEIRO

- O bloco inteiro é predito em blocos do tamanho do tile do `Format` (o `shape` do `DataBase.csv`), pulando os inlines que viraram tile (um bloco a cada `step`): o IoU é só do que a rede não viu.
- Cada bloco passa pela mesma `Normalization` do `Format` (o `scaling` do modelo) e pela predição do treino (`Transforms.infer`: direta, ou por janela deslizante se o modelo treinou em recorte), com o mesmo `amp`.
- Sai o `marlim_iou` por classe (com a classe 0) no `info.json` do modelo e o `marlim_classes.png`.

In [ ]:
class PredictBuilder:
    def __init__(self, target_size, classes, normalize=None, amp=False, save_path=None):
        self.tz, self.ty, self.tx = target_size
        self.classes   = classes
        self.normalize = normalize or Normalization()
        self.amp       = amp
        self.save_path = save_path

    def process(self, vol):
        pads = [(0, (s - vol.shape[i] % s) % s) for i, s in enumerate((self.tz, self.ty, self.tx))]
        if all(p == 0 for _, p in pads): return vol
        return np.pad(vol, pads, mode='reflect')

    def update(self):
        network.model.eval()
        self.pred  = np.zeros(seismic.shape, dtype=np.uint8)
        self.inter = np.zeros(self.classes, dtype=np.int64)
        self.union = np.zeros(self.classes, dtype=np.int64)

        for z in tqdm(range(0, seismic.shape[0], self.tz)):
            # AS FATIAS DE TREINO (UMA A CADA STEP, COMO O Format CORTA) FICAM DE FORA: O IOU E SO DO QUE A REDE NAO VIU
            if (z // self.tz) % STEP == 0:
                continue

            src = np.asarray(seismic[z:z + self.tz], dtype=np.float32)
            if src.shape[0] == 0: break

            img = self.process(src)
            res = np.zeros(img.shape, dtype=np.uint8)

            for y in range(0, img.shape[1], self.ty):
                for x in range(0, img.shape[2], self.tx):
                    # O BLOCO PASSA PELA MESMA Normalization DOS TILES DO Format
                    it = self.normalize(img[:, y:y + self.ty, x:x + self.tx])

                    imgTensor = torch.from_numpy(np.ascontiguousarray(it, dtype=np.float32)).unsqueeze(0).unsqueeze(0).to(network.device)
                    with torch.no_grad(), torch.autocast('cuda', enabled=self.amp):
                        logits = transforms.infer(network.model, imgTensor)
                        pred = torch.argmax(logits, dim=1).squeeze(0) if network.multiclass else (torch.sigmoid(logits) > 0.5).squeeze(0).squeeze(0).int()

                    res[:, y:y + self.ty, x:x + self.tx] = pred.cpu().numpy()

            orig_z = src.shape[0]
            block  = res[:orig_z, :seismic.shape[1], :seismic.shape[2]]
            self.pred[z:z + orig_z] = block

            # iou incremental do volume inteiro (inclui a classe 0)
            gt = np.asarray(masks[z:z + orig_z]).astype(np.uint8)
            for c in range(self.classes):
                pm, gm = (block == c), (gt == c)
                self.inter[c] += np.count_nonzero(pm & gm)
                self.union[c] += np.count_nonzero(pm | gm)

        if self.save_path:
            np.save(self.save_path, self.pred)

    def iou(self):
        vals = self.inter / np.maximum(self.union, 1)
        data = {str(c): float(v) for c, v in enumerate(vals)}
        data['mean'] = float(vals.mean())
        return data


RAW_DIR      = f'../Dataset/{DATASET or OPTIONS.get("dataset")}/files/'
SEISMIC_PATH = os.path.join(RAW_DIR, 'marlim_norm_perc_1-99_cut.npy')
MASK_PATH    = os.path.join(RAW_DIR, 'npy_multiclass_cut.npy')
SAVE_PRED    = False   # opcional: salva predicted.npy (~3.5 GB) no backup
TILE         = ast.literal_eval(df['shape'].iloc[0])

seismic = np.load(SEISMIC_PATH, mmap_mode='r')
masks   = np.load(MASK_PATH, mmap_mode='r')

builder = PredictBuilder(TILE, network.classes, normalize=Normalization(SCALING), amp=OPTIONS.get('amp', False), save_path=os.path.join(baseDir, 'predicted.npy') if SAVE_PRED else None)
builder.update()
builder.pred.shape

In [ ]:
from utils.Plotter.index import Plotter

marlimIoU = builder.iou()
modelInfo['marlim_iou'] = marlimIoU

with open(f'{baseDir}/info.json', 'w', encoding='utf-8') as f:
    json.dump(modelInfo, f, ensure_ascii=False, indent=4)

# Gráfico por classe do bloco inteiro (ordenado por índice da classe: 0 a N, 'mean' no final)
class_keys = sorted([k for k in marlimIoU.keys() if k != 'mean'], key=lambda x: int(x) if x.isdigit() else x)
ious = {k: marlimIoU[k] for k in class_keys}
if 'mean' in marlimIoU:
    ious['mean'] = marlimIoU['mean']

plt.figure(figsize=(17, 6))
Plotter(ious)
plt.savefig(os.path.join(baseDir, 'marlim_classes.png'))
plt.show()

# COMPARAÇÃO VISUAL

In [ ]:
def plotViews(img, mask, pred, alpha=0.5, saveDir=None):
    from matplotlib.patches import Patch
    mx = img.shape[0] // 2
    ig = np.array(img[mx, :, :])
    mg = np.array(mask[mx, :, :])
    pg = np.array(pred[mx, :, :])
    solid_cmap, cmap = faciesCmaps(network.classes)

    for name, layer in [('sismica', None), ('real', mg), ('pred', pg)]:
        plt.figure(figsize=(20, 10))
        plt.imshow(ig, cmap='gray')

        if layer is not None:
            plt.imshow(layer, cmap=cmap, vmin=0, vmax=network.classes - 1, alpha=alpha)
            legend_elements = [Patch(facecolor=solid_cmap.colors[i], edgecolor='white', label=f'Classe {i}') for i in range(network.classes)]
            plt.legend(handles=legend_elements, loc='lower right', fontsize=12)

        plt.title(f"{name.capitalize()} - Slice X={mx}"); plt.axis('off')
        if saveDir:
            plt.savefig(os.path.join(saveDir, f'volume_{name}.png'), dpi=200, bbox_inches='tight')
        plt.show()


viewsDir = os.path.join(baseDir, 'views')
os.makedirs(viewsDir, exist_ok=True)

tz   = builder.tz
ix   = (seismic.shape[0] // 2) // tz * tz
ix   = (ix + tz) if (ix // tz) % STEP == 0 else ix   # fatia do meio que a rede nao viu (as de treino ficam sem predicao)
img  = np.asarray(seismic[ix:ix+tz], dtype=np.float32)
mask = np.asarray(masks[ix:ix+tz], dtype=np.uint8)
pred = builder.pred[ix:ix+tz]
plotViews(img, mask, pred, saveDir=viewsDir)